## Data Loading 

In [4]:
import pandas as pd

df = pd.read_csv("../data/clean_credit_applications.csv")

df.head()

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000.0,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032,78000.0,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000.0,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000.0,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080,57000.0,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN


## Executive Summary

The previous analysis identified significant bias in NovaCred’s credit approval system, including gender, age, and geographic location disparities. Rejections are primarily justified using a generic “algorithm_risk_score,” which does not explain how decisions are made. These findings raise concerns about fairness, transparency, and accountability.

The purpose of this section is to evaluate these issues from a governance perspective by mapping them to GDPR and EU AI Act requirements. This includes assessing privacy risks, transparency gaps, human oversight deficiencies, and overall regulatory exposure, and proposing concrete governance controls to mitigate these risks.

 # Privacy and Governance Gaps 

## 1. Sensitive PII stored without protection

To understand how sensitive personal data is handled, the dataset was examined to identify what personal information is stored and whether it is protected.

In [5]:
df[[
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address"
]].head()

,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address
0,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155
1,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112
2,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250
3,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67
4,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105


The dataset stores full names, email addresses, Social Security Numbers (SSNs), and IP addresses in plain text format. These fields directly identify individuals and are not hashed or encrypted. This Personally Identifiable Information increases the risk of identity theft or fraud. From a governance perspective, there is no visible evidence of technical protection measures embedded at the data level, the dataset itself does not demonstrate privacy-by-design safeguards. This suggests weak implementation of preventive data protection controls.

Mapping to GDPR - This raises concerns under:

- Article 5(1)(c) – Data Minimization: As it is unclear whether storing full SSNs and IP addresses is strictly necessary for credit scoring.

- Article 32 – Security of Processing: Which requires appropriate technical measures to ensure confidentiality and integrity of personal data.

- Article 25 – Data Protection by Design and by Default: Which requires organisations to integrate data protection safeguards into system architecture from the outset.

AI Act Context:

As a high-risk AI credit scoring system, NovaCred is expected to implement robust data governance practices. Weak protection of sensitive personal data undermines compliance with the AI Act’s risk management requirements.

Recommendation:

Sensitive identifiers such as SSNs should be pseudonymized (hashed) or encrypted at rest. Access should be restricted through role-based controls, and a necessity assessment should determine whether storing certain identifiers is required at all.

## Demonstration of Pseudonymization (SSN Hashing)

In [6]:
import hashlib

def hash_value(value):
    return hashlib.sha256(str(value).encode()).hexdigest()

# Create new pseudonymized SSN column
df["pseudonymized_ssn"] = df["applicant_info.ssn"].apply(hash_value)

df[["applicant_info.ssn", "pseudonymized_ssn"]].head()

,applicant_info.ssn,pseudonymized_ssn
0,596-64-4340,2caf30528c21a10e1307b27f9dbbfc312f0c00d46b333e...
1,425-69-4784,2f7da45fefdcfb2c5b4f5b6f1465c22054c36e04fc77c1...
2,370-78-5178,db120edcee2366a48d6d77c2db8c64c5536b8dc3c3c524...
3,194-35-1833,c835719be02018987096d6e49529a24b1d7e7ab35c84b1...
4,480-41-2475,41c7de40dc49185886e6ecb37346ec9eabce16087b7508...


The SSN field has been pseudonymized using a SHA-256 hashing function. This process converts the original identifier into a secure, irreversible string while maintaining consistency across records. In other words, the same SSN will always produce the same hashed value, but the original identifier cannot be directly reconstructed from the hash.

This approach reduces the risk of direct identification in the event of unauthorized access and demonstrates how sensitive identifiers can be protected while preserving their analytical utility. From a governance perspective, pseudonymization supports GDPR requirements related to Security of Processing and Data Protection by Design, as it reduces the exposure of highly sensitive personal data within the system.

However, pseudonymized data remains classified as personal data under the GDPR if it can potentially be re-linked to an individual using additional information. Therefore, pseudonymization alone is not sufficient to eliminate regulatory obligations, the company must also control who can access the data and ensure that stored information is properly protected.

Furthermore, while hashing improves protection, stronger implementations may incorporate techniques such as salting to prevent reverse-engineering or dictionary-based attacks. This highlights that pseudonymization is a risk-reduction measure rather than a complete privacy solution.